#### Basic detector using RegEx (re)
Uses \b word boundaries so bomb matches in “the bomb exploded” but doesn’t match “Bombardier”.
Case-insensitive via re.IGNORECASE.
Returns exact match positions, handy for UI highlighting or logging.
Provides a sanitization function that replaces hits with [REDACTED]

In [1]:

import re
from typing import List, Tuple

def build_blacklist_pattern(keywords: List[str]) -> re.Pattern:
    """
    Build a case-insensitive regex pattern that matches any blacklisted keyword as a whole word.
    Example: ['kill', 'bomb', 'delete', 'drop']
    """
    # Escape keywords for regex safety and join with alternation
    escaped = [re.escape(k) for k in keywords]
    # \b ensures word-boundary matching to avoid 'skill' matching 'kill'
    pattern_str = r"\b(" + "|".join(escaped) + r")\b"
    return re.compile(pattern_str, flags=re.IGNORECASE)

def find_blacklisted(prompt: str, pattern: re.Pattern) -> List[Tuple[str, Tuple[int, int]]]:
    """
    Return a list of (matched_word, (start, end)) for all blacklisted occurrences in the prompt.
    """
    return [(m.group(0), m.span()) for m in pattern.finditer(prompt)]

def is_prompt_allowed(prompt: str, keywords: List[str]) -> bool:
    """
    High-level check: returns True if no blacklisted keywords are found.
    """
    pattern = build_blacklist_pattern(keywords)
    return not bool(pattern.search(prompt))

def sanitize_prompt(prompt: str, keywords: List[str], replacement: str = "[REDACTED]") -> str:
    """
    Replace blacklisted words with a replacement token.
    """
    pattern = build_blacklist_pattern(keywords)
    return pattern.sub(replacement, prompt)

if __name__ == "__main__":
    blacklist = ["kill", "bomb", "delete", "drop"]

    tests = [
        "Please delete the old files.",
        "We plan to drop version 2.0 next week.",
        "He will skill up on Python.",               # 'skill' should NOT be flagged
        "Discuss the bomb disposal procedure.",
        "The KILL switch was engaged.",              # case-insensitive
        "Welcome to Bombardier HQ.",                 # should NOT match 'bomb' due to boundaries
    ]

    pattern = build_blacklist_pattern(blacklist)

    for t in tests:
        matches = find_blacklisted(t, pattern)
        print(f"Prompt: {t}")
        if matches:
            print("  Blacklisted found:", matches)
            print("   Sanitized:", sanitize_prompt(t, blacklist))
        else:
            print("   Allowed (no blacklist hits)")
        print()


Prompt: Please delete the old files.
  Blacklisted found: [('delete', (7, 13))]
   Sanitized: Please [REDACTED] the old files.

Prompt: We plan to drop version 2.0 next week.
  Blacklisted found: [('drop', (11, 15))]
   Sanitized: We plan to [REDACTED] version 2.0 next week.

Prompt: He will skill up on Python.
   Allowed (no blacklist hits)

Prompt: Discuss the bomb disposal procedure.
  Blacklisted found: [('bomb', (12, 16))]
   Sanitized: Discuss the [REDACTED] disposal procedure.

Prompt: The KILL switch was engaged.
  Blacklisted found: [('KILL', (4, 8))]
   Sanitized: The [REDACTED] switch was engaged.

Prompt: Welcome to Bombardier HQ.
   Allowed (no blacklist hits)

